# Daume et al. (2024) on dandiset 000673: PAC neurons

**Paper.** Daume J et al. (2024). Control of working memory by phase-amplitude coupling of human
hippocampal neurons. *Nature* 629, 393-401.
[doi:10.1038/s41586-024-07309-z](https://doi.org/10.1038/s41586-024-07309-z).

**Dandiset.** [DANDI:000673](https://dandiarchive.org/dandiset/000673/0.240118.2135), version
0.240118.2135, the version the paper cites (reference 59); see the header of
`reproduce_daume_2024.ipynb`.

**Text the claims were written from.** The version of record (PMC11078732) with its Methods,
legends and Supplementary Information, and the paper's code (SBCAT-release-NWB at a25e8cc3).

This notebook selects PAC neurons (Methods, Selection of PAC neurons; `sbcat.py` says where each
parameter comes from) and tests the claims on them that need only spikes and the LFP of their
bundle. Artefact rejection uses reading (a) of O1 (on the epochs), the reading adopted in
`reproduce_daume_2024_lfp.ipynb`.

| Claim | Tested here |
|---|---|
| pac-neuron-counts | yes |
| pac-category-overlap | yes |
| fig4c-pac-neuron-firing-single-trial-pac | yes |
| fig4e-hippocampus-pac-neurons-accuracy | yes |
| fig4g-hippocampus-pac-neurons-load | yes |
| fig4-amygdala-pac-neurons | yes |
| SFC, noise correlation and decoding claims on PAC neurons | blocked (attempt record) |

In [ ]:
import os
for _k in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_k] = "1"
import time
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

import sbcat

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore")

started = time.time()
S = sbcat.load_all(sbcat.list_assets())
print(f"loaded trials and units of {len(S)} files in {time.time() - started:.0f} s")
t0 = time.time()
cat_table, _ = sbcat.category_table(S, seed=103)      # the same selection and seed as reproduce_daume_2024.ipynb
print(f"category selection in {time.time() - t0:.0f} s")
t0 = time.time()
P = sbcat.run_all_pac_neurons(S, seed=401, workers=6)
pairs = pd.DataFrame([r for p in P for r in p["rows"]])
counts = np.concatenate([p["counts"] for p in P if len(p["counts"])])
print(f"spike counts for {len(pairs)} neuron-channel-load combinations x {counts.shape[1]} pairings in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
mtl_units = pd.DataFrame([{"session": p["session"], "unit": u} for p in P for u in p["mtl_units"]])
mtl_units["area"] = [S[r.session]["units"].area[r.unit] for r in mtl_units.itertuples()]
with_lfp = pairs.groupby(["session", "unit"]).size().index
print("MTL neurons in sessions with LFP:", mtl_units.area.value_counts().to_dict())
avail = pairs.drop_duplicates(["session", "unit"]).area.value_counts()
print("MTL neurons with at least one LFP channel in their bundle:", avail.to_dict(), "(paper: 212 hippocampal, 358 amygdala)")
print("channels per neuron (bundle):", pairs[pairs.load == 1].groupby(["session", "unit"]).size().value_counts().sort_index().to_dict())
print("trials per combination: median %d, range %d to %d; spikes per combination: median %d, combinations with no spike: %d" % (
    pairs.n_trials.median(), pairs.n_trials.min(), pairs.n_trials.max(), pairs.n_spikes.median(), int((pairs.n_spikes == 0).sum())))
print("total MTL neurons (all sessions):", int(cat_table.area.isin(sbcat.MTL).sum()))

## pac-neuron-counts

> 79 of 212 hippocampal neurons (37.29%) and 163 of 358 amygdala neurons (45.53%) qualified as
> PAC neurons, more than expected by chance.

In [ ]:
t0 = time.time()
K, R1, _ = counts.shape
p12 = np.empty((K, R1)); p13 = np.empty((K, R1)); r2 = np.empty((K, R1))
flat = counts.reshape(-1, 20)
chunk = 100000
p12f, p13f, r2f = [], [], []
for i in range(0, len(flat), chunk):
    o = sbcat.pac_neuron_tests(flat[i:i + chunk])
    p12f.append(o["p12"]); p13f.append(o["p13"]); r2f.append(o["r2"])
p12 = np.concatenate(p12f).reshape(K, R1); p13 = np.concatenate(p13f).reshape(K, R1); r2 = np.concatenate(r2f).reshape(K, R1)
print(f"{3 * flat.shape[0]} Poisson GLMs in {time.time() - t0:.0f} s")
empty = counts.sum(2) == 0
p12[empty] = 1; p13[empty] = 1


def fdr_matrix(p, groups):
    """Benjamini-Hochberg within each group of rows, separately for each column (pairing)."""
    q = np.empty_like(p)
    for g in np.unique(groups):
        m = groups == g
        sub = p[m]
        n = sub.shape[0]
        order = np.argsort(sub, axis=0)
        ranked = np.take_along_axis(sub, order, 0) * n / np.arange(1, n + 1)[:, None]
        ranked = np.minimum.accumulate(ranked[::-1], axis=0)[::-1]
        out = np.empty_like(sub)
        np.put_along_axis(out, order, np.minimum(ranked, 1), 0)
        q[m] = out
    return q


readings = {
    "a": (pairs.area + "|" + pairs.load.astype(str)).to_numpy(),                       # all combinations of an area and load
    "b": (pairs.session.astype(str) + "|" + pairs.unit.astype(str) + "|" + pairs.load.astype(str)).to_numpy(),   # a neuron's channels
}
neuron_key = pairs.session.astype(str) + "|" + pairs.unit.astype(str)
sel = {}
rows = []
for rd, groups in readings.items():
    q12, q13 = fdr_matrix(p12, groups), fdr_matrix(p13, groups)
    ok = (q12 < sbcat.PAC_NEURON_ALPHA) & (q13 < sbcat.PAC_NEURON_ALPHA)          # K x pairings
    per_neuron = pd.DataFrame(ok).groupby(neuron_key.to_numpy()).any()             # neurons x pairings
    area_of = pairs.drop_duplicates(["session", "unit"]).set_index(neuron_key.drop_duplicates())["area"]
    cnt = per_neuron.groupby(area_of.reindex(per_neuron.index).to_numpy()).sum()   # areas x pairings
    sel[rd] = (ok[:, 0], per_neuron[0])
    for a in sbcat.MTL:
        obs = int(cnt.loc[a, 0]); null = cnt.loc[a].to_numpy()[1:]
        n_av = int(avail[a])
        rows.append({"FDR": rd, "area": a, "available": n_av, "pac_neurons": obs, "percent": 100 * obs / n_av,
                     "paper": {"hippocampus": 79, "amygdala": 163}[a], "null_mean": null.mean(), "null_max": null.max(),
                     "p_shuffle": (np.sum(null >= obs) + 1) / (len(null) + 1)})
pac_counts = pd.DataFrame(rows)
print(pac_counts.round(4).to_string(index=False))
print("paper: hippocampus 79 of 212 (37.29%), amygdala 163 of 358 (45.53%), P < 0.005 each")
# how the count depends on the threshold (reading a)
for thr in [0.001, 0.01, 0.05]:
    q12, q13 = fdr_matrix(p12[:, :1], readings["a"]), fdr_matrix(p13[:, :1], readings["a"])
    okt = pd.Series(((q12 < thr) & (q13 < thr))[:, 0]).groupby(neuron_key.to_numpy()).any()
    print(f"FDR threshold {thr}:", okt.groupby(area_of.reindex(okt.index).to_numpy()).sum().to_dict())
print("uncorrected share of combinations with p12 < 0.01 and p13 < 0.01: observed %.3f, shuffled %.3f" % (
    ((p12[:, 0] < 0.01) & (p13[:, 0] < 0.01)).mean(), ((p12[:, 1:] < 0.01) & (p13[:, 1:] < 0.01)).mean()))
# Uncorrected counts per area: neurons with at least one combination (channel, load) passing both tests at p < 0.01
okr = pd.DataFrame((p12 < 0.01) & (p13 < 0.01)).groupby(neuron_key.to_numpy()).any()
cu = okr.groupby(area_of.reindex(okr.index).to_numpy()).sum()
for a in sbcat.MTL:
    null = cu.loc[a].to_numpy()[1:]
    print(f"{a}, uncorrected: {int(cu.loc[a, 0])} of {int(avail[a])} neurons; shuffles mean {null.mean():.1f}, max {null.max()}")
print("combinations per neuron (channels x loads): median %d, range %d to %d" % (
    pairs.groupby(["session", "unit"]).size().median(), pairs.groupby(["session", "unit"]).size().min(), pairs.groupby(["session", "unit"]).size().max()))

**Result: partially reproduced.** With FDR over the combinations of an area and load (O9 a), 8 of
269 hippocampal (2.97%) and 5 of 422 amygdala neurons (1.18%) qualify, against 79 of 212 and 163
of 358; with FDR over each neuron's channels (b), 12 and 24. Both counts exceed every shuffled
count but one (p = 0.005 in both areas under both readings), so the selection finds more than
chance, but the counts are far out of tolerance. Even without any correction, 41 hippocampal and
66 amygdala neurons have a combination passing both tests at p < 0.01 (shuffles at most 13 and
14), fewer than the paper's counts after FDR. The paper's plot_PAC_neuron.m fits the three GLMs to
the 20 aggregated bin counts of one neuron-channel pair, with no random effect (fitglme with
fixed effects only), as `sbcat.py` does, so neither a per-trial layout nor a random effect is the
difference.

In [ ]:
# PAC neurons under FDR reading (a), with the channel of highest model 1 R2 among their qualifying combinations
ok_a = sel["a"][0]
q = pairs.assign(ok=ok_a, r2=r2[:, 0])
pac = (q[q.ok].sort_values("r2", ascending=False).drop_duplicates(["session", "unit"])
       [["session", "subject", "unit", "area", "channel", "lfp_electrode", "load", "r2"]].reset_index(drop=True))
print("PAC neurons by area:", pac.area.value_counts().to_dict())
cat_key = set(zip(cat_table.session[cat_table.is_category], cat_table.unit[cat_table.is_category]))
pac["is_category"] = [(r.session, r.unit) in cat_key for r in pac.itertuples()]

## pac-category-overlap

> The overlap of PAC and category neurons (28 of 79 in the hippocampus, 68 of 163 in the
> amygdala) did not exceed that expected of independent populations.

In [ ]:
analysed = pairs.drop_duplicates(["session", "unit"])[["session", "unit", "area"]]
analysed["is_category"] = [(r.session, r.unit) in cat_key for r in analysed.itertuples()]
rows = []
for a in sbcat.MTL:
    an = analysed[analysed.area == a]
    N, Kc = len(an), int(an.is_category.sum())
    pa = pac[pac.area == a]
    n, k = len(pa), int(pa.is_category.sum())
    p_bin = stats.binom.sf(k - 1, N, (n / N) * (Kc / N))
    p_hyp = stats.hypergeom.sf(k - 1, N, Kc, n)
    rows.append({"area": a, "analysed": N, "category": Kc, "pac": n, "overlap": k, "percent_of_pac": 100 * k / n if n else np.nan,
                 "expected": n * Kc / N, "p_binomial": p_bin, "p_hypergeometric": p_hyp})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("paper: 28 of 79 (35.44%) and 68 of 163 (41.72%), both P > 0.05")

**Result: blocked.** With 8 hippocampal and 5 amygdala PAC neurons (paper 79 and 163), the
overlap (4 of 8 and 2 of 5) does not test the paper's populations; the tests are printed above.

## Firing of PAC neurons

Rates as in `reproduce_daume_2024.ipynb`: maintenance 0 to 2.5 s, baseline -0.9 to -0.3 s before
picture 1 (mean over correct trials); readings of O6 (a) Hz, (b) % of baseline; neurons with
fewer incorrect trials than 1, 2 or 3 are left out of the correct against incorrect tests (O7).

In [ ]:
pac_rates = sbcat.trial_rates(S, pac.assign(preferred=0))


def nmean(r, mask, reading):
    base = r["base"][r["correct"]].mean()
    v = r["maint"][mask].mean() if mask.sum() else np.nan
    if reading == "b":
        return 100 * (v - base) / base if base > 0 else np.nan
    return v


def bmean(r, reading):
    base = r["base"][r["correct"]].mean()
    return (0.0 if base > 0 else np.nan) if reading == "b" else base


def rate_tests(idx, rng):
    rows = []
    for reading in ["a", "b"]:
        rs = [pac_rates[i] for i in idx]
        m_all = np.array([nmean(r, r["correct"], reading) for r in rs])
        b = np.array([bmean(r, reading) for r in rs])
        t = sbcat.perm_ttest_paired(m_all, b, rng)
        rows.append({"reading": reading, "test": "maintenance vs baseline (correct)", "min_inc": "", "n": t["n"], "x": t["mean_x"], "y": t["mean_y"], "t": t["t"], "p": t["p"]})
        for L in (1, 3):
            mL = np.array([nmean(r, r["correct"] & (r["load"] == L), reading) for r in rs])
            t = sbcat.perm_ttest_paired(mL, b, rng)
            rows.append({"reading": reading, "test": f"load {L} vs baseline", "min_inc": "", "n": t["n"], "x": t["mean_x"], "y": t["mean_y"], "t": t["t"], "p": t["p"]})
        m1 = np.array([nmean(r, r["correct"] & (r["load"] == 1), reading) for r in rs])
        m3 = np.array([nmean(r, r["correct"] & (r["load"] == 3), reading) for r in rs])
        t = sbcat.perm_ttest_paired(m3, m1, rng)
        rows.append({"reading": reading, "test": "load 3 vs load 1", "min_inc": "", "n": t["n"], "x": t["mean_x"], "y": t["mean_y"], "t": t["t"], "p": t["p"]})
        fv, sv = [], []
        for r in rs:
            fast, slow = sbcat.rt_split(r["load"], r["rt"], r["correct"])
            fv.append(np.nanmean([nmean(r, fast & (r["load"] == L), reading) for L in (1, 3)]))
            sv.append(np.nanmean([nmean(r, slow & (r["load"] == L), reading) for L in (1, 3)]))
        t = sbcat.perm_ttest_paired(fv, sv, rng)
        rows.append({"reading": reading, "test": "fast vs slow RT", "min_inc": "", "n": t["n"], "x": t["mean_x"], "y": t["mean_y"], "t": t["t"], "p": t["p"]})
        for mi in [1, 2, 3]:
            rk = [r for r in rs if (~r["correct"]).sum() >= mi]
            c = np.array([nmean(r, r["correct"], reading) for r in rk])
            inc = np.array([nmean(r, ~r["correct"], reading) for r in rk])
            bb = np.array([bmean(r, reading) for r in rk])
            for name, x, y in [("correct vs incorrect", c, inc), ("correct vs baseline", c, bb), ("incorrect vs baseline", inc, bb)]:
                t = sbcat.perm_ttest_paired(x, y, rng)
                rows.append({"reading": reading, "test": name, "min_inc": mi, "n": t["n"], "x": t["mean_x"], "y": t["mean_y"], "t": t["t"], "p": t["p"]})
    return pd.DataFrame(rows)

## fig4e-hippocampus-pac-neurons-accuracy and fig4g-hippocampus-pac-neurons-load

> Hippocampal PAC neurons fired above baseline during maintenance, more in correct than in
> incorrect trials, and above baseline in correct trials only.

> Hippocampal PAC neuron rates did not differ between loads but were above baseline in each
> load; they also did not differ between fast and slow RT trials.

In [ ]:
rng = np.random.default_rng(402)
hpc_tests = rate_tests(np.where(pac.area == "hippocampus")[0], rng)
print(f"hippocampal PAC neurons: {int((pac.area == 'hippocampus').sum())} (paper 79)")
print(hpc_tests.round(4).to_string(index=False))
print("paper: maintenance vs baseline t78 = 2.43, p = 0.01; correct vs incorrect t62 = 3.82, p < 0.001 (n = 63); "
      "correct vs baseline t62 = 2.67, p = 0.01; incorrect vs baseline t62 = -0.98, p = 0.33")

In [ ]:
print(hpc_tests[hpc_tests.test.isin(["load 1 vs baseline", "load 3 vs baseline", "load 3 vs load 1", "fast vs slow RT"])].round(4).to_string(index=False))
print("paper: load 3 - load 1 t78 = 1.38, p = 0.20; load 1 vs baseline p = 0.03; load 3 vs baseline p = 0.01; fast vs slow t78 = -0.07, p = 0.94")

**Result: blocked.** The tests above are on 8 hippocampal PAC neurons, against the paper's 79, and
are not a test of the paper's population.

## fig4-amygdala-pac-neurons

> Amygdala PAC neurons fired above baseline during maintenance but did not differ between correct
> and incorrect trials, loads, or fast and slow RT trials.

In [ ]:
rng = np.random.default_rng(403)
amy_tests = rate_tests(np.where(pac.area == "amygdala")[0], rng)
print(f"amygdala PAC neurons: {int((pac.area == 'amygdala').sum())} (paper 163)")
print(amy_tests.round(4).to_string(index=False))
print("paper: maintenance vs baseline t162 = 6.40, p < 0.001; correct vs incorrect t156 = -0.77, p = 0.45 (n = 156); "
      "load 3 - load 1 t162 = 0.26, p = 0.80; fast vs slow t162 = 1.87, p = 0.07")

**Result: blocked.** The tests above are on 5 amygdala PAC neurons, against the paper's 163.

## fig4c-pac-neuron-firing-single-trial-pac

> Spike counts of hippocampal PAC neurons were positively related to single-trial
> theta-high-gamma PAC on their selected channel; amygdala PAC neurons showed no relation.

Correct trials, clean for the selected channel; PAC ~ 1 + SC * load + (1 | patient) + (1 |
neuron-channel pair nested in patient) (Supplementary Table S4 and Methods), SC the spike count
in 0 to 2.5 s; single-trial PAC (O5) (a) the trial's MI, raw, (b) z-scored across trials within
the pair.

In [ ]:
st_index = {(p["session"], ci): (p["st_mi"][ci], p["clean"][ci]) for p in P for ci in p["st_mi"]}
glm_rows = []
for a in sbcat.MTL:
    parts = []
    for r in pac[pac.area == a].itertuples():
        s = S[r.session]
        tr = s["trials"]
        st, clean = st_index[(r.session, r.channel)]
        keep = tr.correct.to_numpy() & clean
        t0_ = tr.timestamps_Maintenance.to_numpy()[keep]
        sc = sbcat.count_in(s["spikes"][r.unit], t0_, t0_ + 2.5, closed=True)
        v = st[keep]
        parts.append(pd.DataFrame({"pac_a": v, "pac_b": (v - np.nanmean(v)) / np.nanstd(v), "sc": sc,
                                   "load": tr.load.to_numpy()[keep].astype(str), "patient": r.subject, "pair": f"{r.session}_{r.unit}_{r.channel}"}))
    if not parts:
        glm_rows.append({"area": a, "pairs": 0})
        continue
    df = pd.concat(parts).dropna().reset_index(drop=True)
    for o5 in ["a", "b"]:
        m = smf.mixedlm(f"pac_{o5} ~ sc * load", df, groups="patient", vc_formula={"pair": "0 + C(pair)"}).fit(reml=True)
        glm_rows.append({"area": a, "O5": o5, "pairs": df.pair.nunique(), "trials": len(df), "estimate": m.params["sc"],
                         "t": m.tvalues["sc"], "p": m.pvalues["sc"], "converged": m.converged})
print(pd.DataFrame(glm_rows).to_string(index=False))
print("paper (Table S4): hippocampus estimate 0.0032, t = 2.19, p = 0.028 (n = 79); amygdala estimate -3e-5, p = 0.98 (n = 163)")

**Result: blocked.** The fits above use 8 hippocampal and 5 amygdala neuron-channel pairs, against
the paper's 79 and 163.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| pac-neuron-counts | partially reproduced (8 and 5 PAC neurons against 79 and 163) |
| pac-category-overlap | blocked (too few PAC neurons) |
| fig4c-pac-neuron-firing-single-trial-pac | blocked (too few PAC neurons) |
| fig4e-hippocampus-pac-neurons-accuracy | blocked (too few PAC neurons) |
| fig4g-hippocampus-pac-neurons-load | blocked (too few PAC neurons) |
| fig4-amygdala-pac-neurons | blocked (too few PAC neurons) |

How the analysis differs from the paper's: the selection follows the paper's plot_PAC_neuron.m
(20 bins, three Poisson GLMs, likelihood-ratio tests); the artefact rejection is the automated
part only (reading a), and the FDR family is not stated (two readings shown). What was found about
the dataset: 269 hippocampal and 422 amygdala neurons have LFP in their bundle, more than the
paper's 212 and 358, so the files do not say which LFPs the paper removed.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")